# Wearly — Makeup & Jewelry AI Fine-Tuning
**Model:** Gemma 4 (via Unsloth)  
**Datasets:** YouMakeup · FFHQ-Makeup · BeautyBank · LADN · CPM · FLUX-HQMT  
**Target capability:** lipstick colour matching · jewelry suggestions · makeup tutorial reasoning

Run on **Google Colab Pro+** (A100 40GB) or local GPU with ≥24 GB VRAM.

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q unsloth datasets transformers trl peft accelerate bitsandbytes
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

In [ ]:
# ── 2. Load base model with Unsloth 4-bit quantisation ───────────────────────
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = 'google/gemma-4-2b-it',  # or gemma-4-9b-it for higher quality
    max_seq_length = MAX_SEQ_LEN,
    dtype        = None,   # auto
    load_in_4bit = True,
)

print('✅ Model loaded:', model.config.model_type)

In [ ]:
# ── 3. Add LoRA adapters (makeup + jewelry domain) ───────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r              = 32,           # rank — higher = more capacity
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    lora_alpha     = 64,
    lora_dropout   = 0.05,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
print('✅ LoRA adapters attached')

In [ ]:
# ── 4. Load and merge all makeup datasets ────────────────────────────────────
import json
from datasets import load_dataset, Dataset, concatenate_datasets
from pathlib import Path

HF_TOKEN = ''   # paste your HuggingFace token if needed for gated datasets

# Wearly instruction format
SYSTEM_PROMPT = """You are Wearly's AI beauty stylist. Given an outfit, weather, and the user's
makeup/jewelry items, suggest personalised lip shades, eye looks, blush, skin finish,
and jewelry. Return JSON with from_wardrobe and buy_suggestions sections."""

def row_to_prompt(row: dict) -> str:
    """Convert any dataset row into Gemma chat format."""
    instruction = row.get('instruction', 'Describe this makeup look.')
    inp         = row.get('input', '')
    output      = row.get('output', '')
    user_msg    = f"{instruction}\n{inp}" if inp else instruction
    return (
        f"<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n"
        f"<start_of_turn>user\n{user_msg}<end_of_turn>\n"
        f"<start_of_turn>model\n{output}<end_of_turn>"
    )

def load_jsonl(path: str) -> Dataset:
    rows = [json.loads(l) for l in open(path)]
    return Dataset.from_list(rows)

datasets_to_merge = []

# ── YouMakeup ───────────────────────────────────────────────────────
try:
    ym = load_dataset('AIM3-RUC/YouMakeup', split='train', token=HF_TOKEN)
    def fmt_youmakeup(row):
        steps = row.get('steps') or row.get('annotations') or []
        step_text = '\n'.join(f"{i+1}. {s}" for i, s in enumerate(steps)) if steps else 'No steps available'
        return {'text': row_to_prompt({'instruction': 'Describe the makeup steps for this tutorial.', 'input': row.get('title',''), 'output': step_text})}
    ym = ym.map(fmt_youmakeup, remove_columns=ym.column_names)
    datasets_to_merge.append(ym)
    print(f'✅ YouMakeup: {len(ym):,} examples')
except Exception as e:
    print(f'⚠️  YouMakeup skipped: {e}')

# ── FFHQ-Makeup ─────────────────────────────────────────────────────
try:
    ffhq = load_dataset('RoyalGentleman/FFHQ-Makeup', split='train', token=HF_TOKEN)
    def fmt_ffhq(row):
        style = row.get('makeup_style') or row.get('style_label','neutral')
        desc  = row.get('description', f'A {style} makeup look.')
        return {'text': row_to_prompt({'instruction': 'Describe this makeup look and how to recreate it.', 'input': f'Style: {style}', 'output': desc})}
    ffhq = ffhq.map(fmt_ffhq, remove_columns=ffhq.column_names)
    datasets_to_merge.append(ffhq)
    print(f'✅ FFHQ-Makeup: {len(ffhq):,} examples')
except Exception as e:
    print(f'⚠️  FFHQ-Makeup skipped: {e}')

# ── BeautyBank ──────────────────────────────────────────────────────
try:
    bb = load_dataset('BUAADreamer/BeautyBank', split='train', token=HF_TOKEN)
    def fmt_bb(row):
        style = row.get('style','unknown')
        similar = row.get('similar_styles',[])
        return {'text': row_to_prompt({'instruction': 'What makeup styles are similar to this one?', 'input': f'Style: {style}', 'output': ', '.join(similar) if similar else 'Natural, No-Makeup Makeup'})}
    bb = bb.map(fmt_bb, remove_columns=bb.column_names)
    datasets_to_merge.append(bb)
    print(f'✅ BeautyBank: {len(bb):,} examples')
except Exception as e:
    print(f'⚠️  BeautyBank skipped: {e}')

# ── CPM (Color Pattern Makeup) ──────────────────────────────────────
try:
    cpm = load_dataset('Qbeast/CPM-Makeup-Dataset', split='train', token=HF_TOKEN)
    def fmt_cpm(row):
        color     = row.get('lip_color') or row.get('color_name','Nude')
        hex_val   = row.get('hex') or row.get('color_hex','#C8907A')
        outfit    = row.get('outfit_color','neutral')
        output    = json.dumps({'lips': {'shade': color, 'hex': hex_val, 'why': f'{color} pairs beautifully with a {outfit} outfit.'}})
        return {'text': row_to_prompt({'instruction': 'Suggest a lipstick color for this outfit.', 'input': f'Outfit colour: {outfit}', 'output': output})}
    cpm = cpm.map(fmt_cpm, remove_columns=cpm.column_names)
    datasets_to_merge.append(cpm)
    print(f'✅ CPM: {len(cpm):,} examples')
except Exception as e:
    print(f'⚠️  CPM skipped: {e}')

# ── FLUX-HQMT ───────────────────────────────────────────────────────
try:
    flux = load_dataset('xiaoming040504/FLUX-Makeup', split='train', token=HF_TOKEN)
    def fmt_flux(row):
        desc = row.get('caption') or row.get('description','Professional makeup look.')
        return {'text': row_to_prompt({'instruction': 'Describe this makeup look in detail.', 'input': '', 'output': desc})}
    flux = flux.map(fmt_flux, remove_columns=flux.column_names)
    datasets_to_merge.append(flux)
    print(f'✅ FLUX-HQMT: {len(flux):,} examples')
except Exception as e:
    print(f'⚠️  FLUX-HQMT skipped: {e}')

# ── Also load Wearly's own /api/learn training data ──────────────────
wearly_jsonl = Path('../datasets/wearly_makeup_train.jsonl')
if wearly_jsonl.exists():
    wearly_ds = load_jsonl(str(wearly_jsonl))
    wearly_ds = wearly_ds.map(lambda r: {'text': row_to_prompt(r)}, remove_columns=wearly_ds.column_names)
    datasets_to_merge.append(wearly_ds)
    print(f'✅ Wearly own data: {len(wearly_ds):,} examples')

# Merge all
from datasets import concatenate_datasets
combined = concatenate_datasets(datasets_to_merge)
combined = combined.shuffle(seed=42)
print(f'\n📊 Total training examples: {len(combined):,}')

In [ ]:
# ── 5. Tokenise ──────────────────────────────────────────────────────────────
def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation    = True,
        max_length    = MAX_SEQ_LEN,
        padding       = False,
    )

tokenised = combined.map(tokenize, batched=True, remove_columns=['text'])
print('✅ Tokenisation done')

In [ ]:
# ── 6. Train ─────────────────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model     = model,
    tokenizer = tokenizer,
    train_dataset    = tokenised,
    dataset_text_field = 'text' if 'text' in combined.column_names else None,
    max_seq_length   = MAX_SEQ_LEN,
    dataset_num_proc = 4,
    args = TrainingArguments(
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 8,    # effective batch = 16
        warmup_steps                 = 100,
        num_train_epochs             = 2,
        learning_rate                = 2e-4,
        fp16                         = not torch.cuda.is_bf16_supported(),
        bf16                         = torch.cuda.is_bf16_supported(),
        logging_steps                = 25,
        save_steps                   = 500,
        output_dir                   = './wearly-makeup-checkpoints',
        optim                        = 'adamw_8bit',
        weight_decay                 = 0.01,
        lr_scheduler_type            = 'cosine',
        seed                         = 42,
    ),
)

print('🚀 Starting training …')
trainer.train()

In [ ]:
# ── 7. Save model in GGUF format for Ollama ──────────────────────────────────
print('💾 Saving model …')

# Save LoRA adapters
model.save_pretrained('wearly-makeup-v1')
tokenizer.save_pretrained('wearly-makeup-v1')

# Export to GGUF (Q4_K_M quantisation — best quality/size balance for Ollama)
model.save_pretrained_gguf(
    'wearly-makeup-v1-gguf',
    tokenizer,
    quantization_method = 'q4_k_m',
)
print('✅ GGUF saved → wearly-makeup-v1-gguf/')

In [ ]:
# ── 8. Push to HuggingFace Hub (optional) ────────────────────────────────────
HF_REPO = 'harsaikron/wearly-makeup-v1'   # change to your org/username

model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f'✅ Model pushed → https://huggingface.co/{HF_REPO}')

## Load into Ollama (run on your Mac)
```bash
# Copy the .gguf file to your Mac
scp wearly-makeup-v1-gguf/unsloth.Q4_K_M.gguf ~/

# Create Modelfile
cat > ~/Modelfile.makeup << 'EOF'
FROM ~/unsloth.Q4_K_M.gguf
SYSTEM """You are Wearly's AI beauty stylist. Given an outfit, weather, and the user's
makeup/jewelry items, suggest personalised lip shades, eye looks, blush, skin finish,
and jewelry. Return valid JSON only."""
PARAMETER temperature 0.7
PARAMETER num_ctx 4096
EOF

# Register with Ollama
ollama create wearly-makeup-v1 -f ~/Modelfile.makeup

# Test it
ollama run wearly-makeup-v1 "Suggest a lip colour for a navy shirt in 32°C Singapore heat."
```